# AlexNet on CIFAR-10

This notebook provides an implementation of **AlexNet** (Krizhevsky et al., 2012), adapted for the CIFAR-10 dataset. 

It demonstrates the process of building the model architecture, preparing the dataset, training the model, and evaluating its performance.

## 1. Imports and Setup
First, we import the necessary PyTorch modules and configure the device.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Dataset Preparation
AlexNet was originally designed for ImageNet ($224 \times 224$ images). CIFAR-10 images are $32 \times 32$. To use the standard AlexNet architecture, we resize the CIFAR-10 images to $227 \times 227$ (which fits the convolutions properly).

In [ ]:
transform = transforms.Compose([
    transforms.Resize(227),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

batch_size = 64

train_data = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Testing batches: {len(test_loader)}")

## 3. Model Architecture
We define the `AlexNet` class following the original paper, composed of a feature extractor (convolutions and max pooling) and a classifier (fully connected layers with dropout).

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10) -> None:
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = AlexNet(num_classes=10).to(device)
print(model)

In [ ]:
# Model Summary

from torchsummary import summary

summary(model=model, input_size=(3, 227, 227))

## 4. Training the Model
Set up the loss function (CrossEntropyLoss) and the optimizer (SGD with momentum).

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=0.0005)

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

    print(f"==> Epoch {epoch+1} Average Loss: {total_loss/len(train_loader):.4f}")

# Save the trained weights
torch.save(model.state_dict(), 'alexnet.pth')

In [ ]:
# model.load_state_dict(torch.load("alexnet.pth"))
# model.to(device)

## 5. Evaluation
Calculate the accuracy on the test set.

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

## 6. Predict Single Image

In [ ]:
import matplotlib.pyplot as plt

classes = train_data.classes

img, label = test_data[0]

model.eval()

with torch.no_grad():
    output = model(img.unsqueeze(0).to(device))
    _, pred = torch.max(output, 1)

plt.imshow(img.permute(1,2,0))
plt.title(f"Pred: {classes[pred.item()]}")
plt.show()